# Test delays transformers

## Importation des modules

In [23]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Importation des modules
# Modules de base
import pandas as pd
import numpy as np
import sys

sys.path.append('..')

# Modules du package
from tsforecast.delays.auxiliary_transformers import ShiftTransformer, MaskTransformer
from tsforecast.panel import PanelwiseTransformer

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Test sur données de séries temporelles

### Génération des données

In [24]:
# Set random seed for reproducibility
np.random.seed(42)

# Create monthly dates from 2020 to 2024
monthly_dates = pd.date_range('2020-01-01', '2024-01-01', freq='MS')
quarterly_dates = pd.date_range('2020-01-01', '2024-01-01', freq='QS')

def generate_economic_series(dates, base_value=100, trend=0.02, volatility=0.05):
    """Generate realistic economic time series"""
    n = len(dates)
    # Trend component
    trend_component = np.cumsum(np.random.normal(trend/12, volatility/4, n))
    # Cyclical component
    cycle = 0.1 * np.sin(2 * np.pi * np.arange(n) / 12) 
    # Random noise
    noise = np.random.normal(0, volatility, n)
    
    return base_value * np.exp(trend_component + cycle + noise)

# Generate monthly indicators
monthly_data = pd.DataFrame({
    'date': monthly_dates,
    'country': 'US',
    'inflation_rate': generate_economic_series(monthly_dates, 2.0, 0.001, 0.02),
    'unemployment_rate': generate_economic_series(monthly_dates, 5.0, -0.001, 0.03),
    'industrial_production': generate_economic_series(monthly_dates, 100, 0.002, 0.04)
}).set_index('date')

# Generate quarterly indicators
quarterly_data = pd.DataFrame({
    'date': quarterly_dates,
    'country': 'US', 
    'gdp_growth': generate_economic_series(quarterly_dates, 2.5, 0.0005, 0.015),
    'government_debt': generate_economic_series(quarterly_dates, 80, 0.01, 0.02)
}).set_index('date')

print("Monthly data shape:", monthly_data.shape)
print("Quarterly data shape:", quarterly_data.shape)
print("\nMonthly data sample:")
print(monthly_data.head())
print("\nQuarterly data sample:")
print(quarterly_data.head())

Monthly data shape: (49, 4)
Quarterly data shape: (17, 3)

Monthly data sample:
           country  inflation_rate  unemployment_rate  industrial_production
date                                                                        
2020-01-01      US        1.935670           4.805587              97.818941
2020-02-01      US        2.120364           5.329239             102.432535
2020-03-01      US        2.175653           5.432306             105.574865
2020-04-01      US        2.209106           5.480127             116.548335
2020-05-01      US        2.234182           5.408413             109.315070

Quarterly data sample:
           country  gdp_growth  government_debt
date                                           
2020-01-01      US    2.545057        80.690558
2020-04-01      US    2.657792        82.351275
2020-07-01      US    2.765814        85.556844
2020-10-01      US    2.827254        87.593422
2021-01-01      US    2.744863        87.410966


### Test du ShiftTransformer

#### Sur données mensuelles avec fréquence mensuelle

In [25]:
# 
shift_test1 = ShiftTransformer(n_periods=2, frequency="M")

res1 = shift_test1.transform(monthly_data[['inflation_rate', 'unemployment_rate']])

print(res1.head())

print(res1.tail())

            inflation_rate  unemployment_rate
2019-11-01        1.935670           4.805587
2019-12-01        2.120364           5.329239
2020-01-01        2.175653           5.432306
2020-02-01        2.209106           5.480127
2020-03-01        2.234182           5.408413
            inflation_rate  unemployment_rate
2023-07-01        1.742092           4.628184
2023-08-01        1.710669           4.485273
2023-09-01        1.693223           4.359109
2023-10-01        1.829053           4.745601
2023-11-01        1.924947           4.973987


In [26]:
res11 = shift_test1.inverse_transform(res1)

print(res11.head())

print(res11.tail())

            inflation_rate  unemployment_rate
2020-01-01        1.935670           4.805587
2020-02-01        2.120364           5.329239
2020-03-01        2.175653           5.432306
2020-04-01        2.209106           5.480127
2020-05-01        2.234182           5.408413
            inflation_rate  unemployment_rate
2023-09-01        1.742092           4.628184
2023-10-01        1.710669           4.485273
2023-11-01        1.693223           4.359109
2023-12-01        1.829053           4.745601
2024-01-01        1.924947           4.973987


In [27]:
res2 = shift_test1.transform(monthly_data['inflation_rate'])

print(res2.head())

print(res2.tail())

2019-11-01    1.935670
2019-12-01    2.120364
2020-01-01    2.175653
2020-02-01    2.209106
2020-03-01    2.234182
Name: inflation_rate, dtype: float64
2023-07-01    1.742092
2023-08-01    1.710669
2023-09-01    1.693223
2023-10-01    1.829053
2023-11-01    1.924947
Name: inflation_rate, dtype: float64


In [28]:
res21 = shift_test1.inverse_transform(res2)

print(res21.head())

print(res21.tail())

2020-01-01    1.935670
2020-02-01    2.120364
2020-03-01    2.175653
2020-04-01    2.209106
2020-05-01    2.234182
Name: inflation_rate, dtype: float64
2023-09-01    1.742092
2023-10-01    1.710669
2023-11-01    1.693223
2023-12-01    1.829053
2024-01-01    1.924947
Name: inflation_rate, dtype: float64


In [29]:
# 
shift_test2 = ShiftTransformer(n_periods=2, frequency="Q")

res3 = shift_test2.transform(pd.concat([quarterly_data[["gdp_growth", "government_debt"]], monthly_data[['inflation_rate', 'unemployment_rate']]], axis=1))

print(res3.head())

print(res3.tail())

            gdp_growth  government_debt  inflation_rate  unemployment_rate
2019-07-01    2.545057        80.690558        1.935670           4.805587
2019-08-01         NaN              NaN        2.120364           5.329239
2019-09-01         NaN              NaN        2.175653           5.432306
2019-10-01    2.657792        82.351275        2.209106           5.480127
2019-11-01         NaN              NaN        2.234182           5.408413
            gdp_growth  government_debt  inflation_rate  unemployment_rate
2023-03-01         NaN              NaN        1.742092           4.628184
2023-04-01    2.849417        89.828681        1.710669           4.485273
2023-05-01         NaN              NaN        1.693223           4.359109
2023-06-01         NaN              NaN        1.829053           4.745601
2023-07-01    2.793920        90.310420        1.924947           4.973987


In [30]:
res31 = shift_test2.inverse_transform(res3)

print(res31.head())

print(res31.tail())

            gdp_growth  government_debt  inflation_rate  unemployment_rate
2020-01-01    2.545057        80.690558        1.935670           4.805587
2020-02-01         NaN              NaN        2.120364           5.329239
2020-03-01         NaN              NaN        2.175653           5.432306
2020-04-01    2.657792        82.351275        2.209106           5.480127
2020-05-01         NaN              NaN        2.234182           5.408413
            gdp_growth  government_debt  inflation_rate  unemployment_rate
2023-09-01         NaN              NaN        1.742092           4.628184
2023-10-01    2.849417        89.828681        1.710669           4.485273
2023-11-01         NaN              NaN        1.693223           4.359109
2023-12-01         NaN              NaN        1.829053           4.745601
2024-01-01    2.793920        90.310420        1.924947           4.973987


In [31]:


# Test sfit positive / négative
# Test series/dataframes
# Test fréquence index supérieure à celle de l'indicateur
# Test fréquence de l'indicateur différente de celle du shift
# 

## Test MaskTransformer

In [32]:
# 
mask_test1 = MaskTransformer(n_obs=2, mask_frequency="Q")

res1 = mask_test1.transform(monthly_data[['inflation_rate', 'unemployment_rate']])

print(res1.head())

print(res1.tail())

            inflation_rate  unemployment_rate
date                                         
2020-01-01        1.935670           4.805587
2020-02-01             NaN                NaN
2020-03-01             NaN                NaN
2020-04-01        2.209106           5.480127
2020-05-01             NaN                NaN
            inflation_rate  unemployment_rate
date                                         
2023-09-01             NaN                NaN
2023-10-01        1.710669           4.485273
2023-11-01             NaN                NaN
2023-12-01             NaN                NaN
2024-01-01             NaN                NaN


In [33]:
res11 = mask_test1.inverse_transform(res1)

print(res11.head())

print(res11.tail())

            inflation_rate  unemployment_rate
date                                         
2020-01-01        1.935670           4.805587
2020-02-01        2.120364           5.329239
2020-03-01        2.175653           5.432306
2020-04-01        2.209106           5.480127
2020-05-01        2.234182           5.408413
            inflation_rate  unemployment_rate
date                                         
2023-09-01        1.742092           4.628184
2023-10-01        1.710669           4.485273
2023-11-01        1.693223           4.359109
2023-12-01        1.829053           4.745601
2024-01-01        1.924947           4.973987


In [34]:
res2 = mask_test1.transform(monthly_data['inflation_rate'])

print(res2.head())

print(res2.tail())

date
2020-01-01    1.935670
2020-02-01         NaN
2020-03-01         NaN
2020-04-01    2.209106
2020-05-01         NaN
Name: inflation_rate, dtype: float64
date
2023-09-01         NaN
2023-10-01    1.710669
2023-11-01         NaN
2023-12-01         NaN
2024-01-01         NaN
Name: inflation_rate, dtype: float64


In [35]:
res21 = mask_test1.inverse_transform(res2)

print(res21.head())

print(res21.tail())

date
2020-01-01    1.935670
2020-02-01    2.120364
2020-03-01    2.175653
2020-04-01    2.209106
2020-05-01    2.234182
Name: inflation_rate, dtype: float64
date
2023-09-01    1.742092
2023-10-01    1.710669
2023-11-01    1.693223
2023-12-01    1.829053
2024-01-01    1.924947
Name: inflation_rate, dtype: float64


In [36]:
# 
mask_test2 = MaskTransformer(n_obs=2, mask_frequency="M")

res2 = mask_test2.transform(monthly_data[['inflation_rate', 'unemployment_rate']])

print(res2.head())

print(res2.tail())

ValueError: The index frequency should be strictly higher than the mask frequency.The index frequency is M and the mask frequency is M

## Test des transformers sur des données de panel

### Génération de données de Panel

In [ ]:
# Generate monthly indicators
monthly_panel_data = pd.DataFrame({
    'date': monthly_dates,
    'country': 'US',
    'inflation_rate': generate_economic_series(monthly_dates, 2.0, 0.001, 0.02),
    'unemployment_rate': generate_economic_series(monthly_dates, 5.0, -0.001, 0.03),
    'industrial_production': generate_economic_series(monthly_dates, 100, 0.002, 0.04)
}).set_index(['country', 'date'])

# Generate quarterly indicators
quarterly_panel_data = pd.DataFrame({
    'date': quarterly_dates,
    'country': 'US', 
    'gdp_growth': generate_economic_series(quarterly_dates, 2.5, 0.0005, 0.015),
    'government_debt': generate_economic_series(quarterly_dates, 80, 0.01, 0.02)
}).set_index(['country', 'date'])

print("Monthly data shape:", monthly_panel_data.shape)
print("Quarterly data shape:", quarterly_panel_data.shape)
print("\nMonthly data sample:")
print(monthly_panel_data.head())
print("\nQuarterly data sample:")
print(quarterly_panel_data.head())

### Test du ShiftTransformer

In [ ]:
panel_shift_transformer = PanelwiseTransformer(
    transformer=ShiftTransformer(n_periods=2, frequency='M'),
    time_col=None,
    panel_cols=None
)

panel_test1 = panel_shift_transformer.fit_transform(monthly_panel_data)

panel_test1.head()

In [ ]:
panel_test11 = panel_shift_transformer.inverse_transform(panel_test1)
panel_test11.head()

### Test du MaskTransformer

In [ ]:
panel_mask_transformer = PanelwiseTransformer(
    transformer=MaskTransformer(n_obs=2, mask_frequency='Q'),
    time_col=None,
    panel_cols=None
)

panel_test2 = panel_mask_transformer.fit_transform(monthly_panel_data)

panel_test2.head()

In [ ]:
panel_test21 = panel_mask_transformer.inverse_transform(panel_test2)
panel_test21.head()

#### Sur données trimestrielle avec fréquence mensuelle

### Test du MaskTransformer

## Test sur données de panel

### Génération des données de panel